In [1]:
!pip install -q langchain-pinecone pinecone langchain-huggingface sentence-transformers


In [2]:
#Getting API key of pinecone at https://app.pinecone.io

import os
from getpass import getpass
os.environ['PINECONE_API_KEY'] = getpass("Pinecone Api Key")


Pinecone Api Key··········


In [3]:
# Creating documents for players

from langchain_core.documents import Document

doc1 = Document(
    page_content = "Babar Azam is a Pakistani right-handed batter from Lahore. "
        "He is known for his elegant cover drive and consistent run-scoring "
        "across Test, ODI and T20 cricket.",
    metadata = {"name": "Babar Azam", "role": "Batter", "city": "Lahore"}
)
doc2 = Document(
    page_content=(
        "Shaheen Shah Afridi is a left-arm fast bowler. "
        "He is known for swinging the new ball and taking early wickets."
    ),
    metadata={"name": "Shaheen Shah Afridi", "role": "Fast bowler", "city": "Landi Kotal"},
)

doc3 = Document(
    page_content=(
        "Mohammad Rizwan is a wicketkeeper-batter from Peshawar. "
        "He is known for his fitness, his composure under pressure and "
        "his reliable batting at the top of the order."
    ),
    metadata={"name": "Mohammad Rizwan", "role": "Wicketkeeper-batter", "city": "Peshawar"},
)

doc4 = Document(
    page_content=(
        "Shadab Khan is a leg-spin all-rounder. "
        "He contributes with the ball in the middle overs, with quick "
        "runs lower down the order and with sharp fielding."
    ),
    metadata={"name": "Shadab Khan", "role": "All-rounder", "city": "Mianwali"},
)

doc5 = Document(
    page_content=(
        "Haris Rauf is a right-arm fast bowler from Rawalpindi. "
        "He is known for his pace and for bowling in the powerplay and "
        "at the death in T20 cricket."
    ),
    metadata={"name": "Haris Rauf", "role": "Fast bowler", "city": "Rawalpindi"},
)

docs = [doc1, doc2, doc3, doc4, doc5]


In [4]:
# free hugging face embeddings (384 dimensions)
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# Create pinecone index (Dimensions must match embedding model)

import time
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key = os.environ["PINECONE_API_KEY"])
index_name = "players"
if index_name not in pc.list_indexes().names():
  pc.create_index(
      name = index_name,
      dimension = 384,
      metric = "cosine",
      spec = ServerlessSpec(cloud ="aws", region= "us-east-1"),

  )

#wait untill index is ready

while not pc.describe_index(index_name).status["ready"]:
  time.sleep(1)


index =pc.Index(index_name)


In [8]:
!pip uninstall -y pinecone-client pinecone
!pip install -q -U pinecone langchain-pinecone

Found existing installation: pinecone 7.3.0
Uninstalling pinecone-7.3.0:
  Successfully uninstalled pinecone-7.3.0


In [9]:
# Create vector store and add the dcuments in it with out own IDs
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding = embeddings)
ids = ["babar", "shaheen", "rizwan", "shadab", "rauf"]
vector_store.add_documents(docs, ids=ids)
time.sleep(5)


In [10]:
# fetching vectors (view documents)
# langchain stores the text in the "text" metadata field

print(index.describe_index_stats())       # how many vectors are stored

res = index.fetch(ids=["babar", "shaheen"])
for id_, vec in res.vectors.items():
    print(id_)
    print("  metadata :", vec.metadata)
    print("  embedding:", vec.values[:5], "... (384 numbers)")

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5}},
 'total_vector_count': 5,
 'vector_type': 'dense'}
babar
  metadata : {'city': 'Lahore', 'name': 'Babar Azam', 'role': 'Batter', 'text': 'Babar Azam is a Pakistani right-handed batter from Lahore. He is known for his elegant cover drive and consistent run-scoring across Test, ODI and T20 cricket.'}
  embedding: [-0.0269615017, 0.0637864694, -0.106951624, 0.0404669568, -0.0795113] ... (384 numbers)
shaheen
  metadata : {'city': 'Landi Kotal', 'name': 'Shaheen Shah Afridi', 'role': 'Fast bowler', 'text': 'Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.'}
  embedding: [0.00958232395, 0.0484874807, -0.0180721451, 0.0318278, 0.00219021738] ... (384 numbers)


In [11]:
#Searching through the documents

vector_store.similarity_search(
    query= "Who is the best player among the team with respect to bowling",
    k=2,
)

[Document(id='shaheen', metadata={'city': 'Landi Kotal', 'name': 'Shaheen Shah Afridi', 'role': 'Fast bowler'}, page_content='Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.'),
 Document(id='shadab', metadata={'city': 'Mianwali', 'name': 'Shadab Khan', 'role': 'All-rounder'}, page_content='Shadab Khan is a leg-spin all-rounder. He contributes with the ball in the middle overs, with quick runs lower down the order and with sharp fielding.')]

In [12]:
# Searching documents, filter on metadata (here role) and get the similarity score
# NOTE: with cosine in Pinecone a HIGHER score is MORE similar (opposite of Chroma distance)

results = vector_store.similarity_search_with_score(
    query="who swings the new ball?",
    k=2,
    filter={"role": "Fast bowler"},
)
for doc, score in results:
    print(doc.metadata["name"], "| score:", round(score, 3))

Shaheen Shah Afridi | score: 0.377
Haris Rauf | score: 0.245


In [13]:
# more filter examples
vector_store.similarity_search(
    "good player",
    k=3,
    filter={"city": {"$in": ["Lahore", "Peshawar"]}},
)

[Document(id='babar', metadata={'city': 'Lahore', 'name': 'Babar Azam', 'role': 'Batter'}, page_content='Babar Azam is a Pakistani right-handed batter from Lahore. He is known for his elegant cover drive and consistent run-scoring across Test, ODI and T20 cricket.'),
 Document(id='rizwan', metadata={'city': 'Peshawar', 'name': 'Mohammad Rizwan', 'role': 'Wicketkeeper-batter'}, page_content='Mohammad Rizwan is a wicketkeeper-batter from Peshawar. He is known for his fitness, his composure under pressure and his reliable batting at the top of the order.')]

In [14]:
# update documents
# Pinecone has no update_document: adding with the SAME id overwrites the record

updated_doc2 = Document(
    page_content=(
        "Naseem Shah is a right-arm fast bowler from Lower Dir. "
        "He is known for his pace and for taking wickets with the new ball "
        "at a young age."
    ),
    metadata={"name": "Naseem Shah", "role": "Fast bowler", "city": "Lower Dir"},
)

vector_store.add_documents([updated_doc2], ids=["shaheen"])
time.sleep(5)

print(index.fetch(ids=["shaheen"]).vectors["shaheen"].metadata)

for r in vector_store.similarity_search("Naseem Shah", k=2):
    print(r.metadata["name"])

{'city': 'Lower Dir', 'name': 'Naseem Shah', 'role': 'Fast bowler', 'text': 'Naseem Shah is a right-arm fast bowler from Lower Dir. He is known for his pace and for taking wickets with the new ball at a young age.'}
Naseem Shah
Mohammad Rizwan


In [15]:
# delete documents
vector_store.delete(ids=["rauf"])
time.sleep(3)
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 4}},
 'total_vector_count': 4,
 'vector_type': 'dense'}


In [16]:
# reconnect to an existing index later (data is still there)
vector_store = PineconeVectorStore(index=pc.Index("players"), embedding=embeddings)
vector_store.similarity_search("opening batter", k=1)

[Document(id='rizwan', metadata={'city': 'Peshawar', 'name': 'Mohammad Rizwan', 'role': 'Wicketkeeper-batter'}, page_content='Mohammad Rizwan is a wicketkeeper-batter from Peshawar. He is known for his fitness, his composure under pressure and his reliable batting at the top of the order.')]

In [17]:
# cleanup: delete the whole index when you are finished
# pc.delete_index("players")